In [1]:

import pandas as pd
import numpy as np
import sqlite3
from flask import Flask, render_template, request
import matplotlib.pyplot as plt
import io
import base64





In [2]:
cdf= pd.read_csv('/Users/user/Desktop/Individual_DE/Website_data/commodity_driven_deforestation.csv', skiprows=3)

In [4]:
cdf.head(10)

,Country_code_A3,Name,Y_1990,Y_1991,Y_1992,Y_1993,Y_1994,Y_1995,Y_1996,Y_1997,...,Y_2009,Y_2010,Y_2011,Y_2012,Y_2013,Y_2014,Y_2015,Y_2016,Y_2017,Y_2018
0,ABW,Aruba,52.467587,58.445779,66.702558,62.301504,68.506353,75.829265,58.734385,81.162564,...,159.364420,141.075838,135.129301,173.603909,169.526161,172.058516,169.902421,172.877400,158.317865,154.667162
1,AFG,Afghanistan,10625.107870,10830.881100,10950.536730,11171.821500,11299.594870,11815.267600,13155.159110,14252.954840,...,18810.398530,20963.797330,21558.233160,21216.818800,20942.206790,21451.730190,21147.538780,21050.778270,21302.708370,21196.132150
2,AGO,Angola,32156.596760,32464.235280,32567.388290,32602.146280,34115.353340,34801.965220,40930.126650,40057.615720,...,65693.104140,67693.467210,66277.366280,67652.087410,67200.277610,68146.581220,69470.341230,69064.557600,71539.342720,66445.751310
3,AIA,Anguilla,2.726608,3.009267,3.130169,3.406827,3.932457,4.488219,4.493260,4.194689,...,5.795507,6.373054,6.660545,6.705420,6.993232,7.027910,7.078796,7.159170,7.272695,7.181544
4,AIR,Int. Aviation,1322.248427,1285.268168,1325.968003,1352.131430,1424.838702,1481.416851,1541.112346,1620.190699,...,2216.413310,2339.399901,2426.222186,2445.366423,2487.015536,2570.114990,2712.218964,2831.883565,2990.773933,3116.257046
5,ALB,Albania,4260.823984,3853.034806,3551.114440,3755.743353,4454.070176,4417.983441,4129.354299,3973.571397,...,4092.736698,4346.630812,4426.243474,4294.442321,4198.182363,4651.494752,4577.566621,4528.350526,4926.751156,5086.417330
6,ANT,Netherlands Antilles,228.094143,235.579196,248.016585,215.260199,243.996526,245.146210,258.167663,434.611107,...,609.244006,459.446646,564.052535,498.128454,464.444263,511.331291,516.270905,460.654061,422.086876,372.206431
7,ARE,United Arab Emirates,7161.870638,7411.507169,7757.682681,8616.151621,8847.981502,9599.888151,10160.410970,10377.816230,...,24620.335190,24944.416870,25559.527990,27527.769420,29265.157470,30907.821750,31827.551880,34871.839290,37305.897380,38592.900720
8,ARG,Argentina,247217.663600,246609.312200,249510.577000,247326.240100,250635.872500,255865.360400,255588.650800,256532.805700,...,334389.433700,329099.059200,295991.083900,298185.857500,305954.807300,312016.358400,313814.954300,273980.076000,276767.152700,274442.274700
9,ARM,Armenia,4676.558361,4040.579411,3129.845981,2349.860662,2191.870103,2177.509163,2181.722893,2206.187664,...,2589.416114,2555.527407,2439.230174,2607.801125,2669.183929,3124.321359,3177.903634,3214.052993,3092.817637,2960.659072


In [9]:
# Renaming the year columns  
cdf.rename(columns={f"Y_{year}": str(year) for year in range(1990, 2019)}, inplace=True)

# Converting to long format
cdf_long = pd.melt(
    cdf,
    id_vars=["Name"],                # or use "Country" if you've renamed it
    value_vars=[str(y) for y in range(1990, 2019)],
    var_name="Year",
    value_name="Deforestation"
)

# Rename column for clarity
cdf_long.rename(columns={"Name": "Country"}, inplace=True)

# Preview
print(cdf_long.head())


         Country  Year  Deforestation
0          Aruba  1990      52.467587
1    Afghanistan  1990   10625.107870
2         Angola  1990   32156.596760
3       Anguilla  1990       2.726608
4  Int. Aviation  1990    1322.248427


In [3]:
ghg = pd.read_csv("/Users/user/Desktop/Individual_DE/Website_data/greenhousegas.csv", skiprows=4)


In [16]:
ghg.head(10)

,Time period,Time period.1,1990,1991,1992,1993,1994,1995,1996,1997,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,Unnamed: 34
0,Reference area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,NaN,"438,056.76","438,049.29","441,752.18","442,282.48","442,610.56","451,076.46","457,574.99","469,835.69",...,"543,798.68","535,451.44","544,231.82","552,354.83","559,581.11","560,827.41","555,244.93","536,739.72","528,631.65",NaN
2,Austria,NaN,"79,047.23","82,711.16","76,142.81","76,517.86","76,725.97","79,953.24","83,112.93","82,719.63",...,"80,228.52","76,662.66","78,884.47","79,821.26","82,132.49","78,854.38","79,994.14","73,910.84","77,532.35",NaN
3,Belgium,NaN,"145,844.47","148,579.58","148,077.75","146,896.93","151,445.45","153,579.83","157,227.95","148,820.67",...,"120,523.72","114,878.85","118,990.38","117,419.57","116,909.24","117,584.92","116,463.71","107,272.65","110,951.73",NaN
4,Canada,NaN,"588,602.82","582,031.22","599,242.66","601,743.65","621,934.13","639,070.29","660,767.80","676,394.22",...,"723,096.30","720,195.46","722,918.27","704,927.87","712,233.80","724,615.78","723,679.29","658,788.39","670,428.31",NaN
5,Chile,NaN,"48,818.66","47,107.52","49,071.29","51,773.06","54,516.64","57,660.90","63,314.63","69,817.16",...,"100,438.19","96,796.60","103,039.02","107,545.47","108,024.61","109,460.81","111,026.60","105,551.92",NaN,NaN
6,Colombia,NaN,"101,878.75","104,424.76","109,571.27","111,662.30","113,044.86","115,599.56","117,221.56","123,168.80",...,"162,655.74","165,839.29","170,637.07","175,628.86","171,035.50","180,727.24",NaN,NaN,NaN,NaN
7,Costa Rica,NaN,"8,196.56","8,056.64","8,797.32","8,892.50","9,622.28","9,693.60","9,562.85","9,272.53",...,"13,735.74","13,850.14","13,603.64","14,234.46","14,477.60",NaN,NaN,NaN,NaN,NaN
8,Czechia,NaN,"198,775.27","180,859.67","175,034.85","168,323.67","159,410.64","158,371.95","161,709.65","157,200.37",...,"129,251.71","127,080.04","128,506.45","130,106.97","130,951.94","129,180.06","123,451.26","113,072.05","118,381.69",NaN
9,Denmark,NaN,"71,820.94","82,433.00","76,517.58","78,670.86","82,598.77","79,748.73","92,928.69","83,411.41",...,"57,779.39","53,608.90","50,834.62","52,906.15","50,773.91","50,726.02","46,941.67","44,483.80","45,515.58",NaN


In [4]:

ghg.drop(columns=[ghg.columns[1]], inplace=True)
ghg.head(10)



,Time period,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2013,2014,2015,2016,2017,2018,2019,2020,2021,Unnamed: 34
0,Reference area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,"438,056.76","438,049.29","441,752.18","442,282.48","442,610.56","451,076.46","457,574.99","469,835.69","484,037.12",...,"543,798.68","535,451.44","544,231.82","552,354.83","559,581.11","560,827.41","555,244.93","536,739.72","528,631.65",NaN
2,Austria,"79,047.23","82,711.16","76,142.81","76,517.86","76,725.97","79,953.24","83,112.93","82,719.63","82,012.90",...,"80,228.52","76,662.66","78,884.47","79,821.26","82,132.49","78,854.38","79,994.14","73,910.84","77,532.35",NaN
3,Belgium,"145,844.47","148,579.58","148,077.75","146,896.93","151,445.45","153,579.83","157,227.95","148,820.67","154,012.89",...,"120,523.72","114,878.85","118,990.38","117,419.57","116,909.24","117,584.92","116,463.71","107,272.65","110,951.73",NaN
4,Canada,"588,602.82","582,031.22","599,242.66","601,743.65","621,934.13","639,070.29","660,767.80","676,394.22","682,434.83",...,"723,096.30","720,195.46","722,918.27","704,927.87","712,233.80","724,615.78","723,679.29","658,788.39","670,428.31",NaN
5,Chile,"48,818.66","47,107.52","49,071.29","51,773.06","54,516.64","57,660.90","63,314.63","69,817.16","70,882.57",...,"100,438.19","96,796.60","103,039.02","107,545.47","108,024.61","109,460.81","111,026.60","105,551.92",NaN,NaN
6,Colombia,"101,878.75","104,424.76","109,571.27","111,662.30","113,044.86","115,599.56","117,221.56","123,168.80","123,130.08",...,"162,655.74","165,839.29","170,637.07","175,628.86","171,035.50","180,727.24",NaN,NaN,NaN,NaN
7,Costa Rica,"8,196.56","8,056.64","8,797.32","8,892.50","9,622.28","9,693.60","9,562.85","9,272.53","9,811.83",...,"13,735.74","13,850.14","13,603.64","14,234.46","14,477.60",NaN,NaN,NaN,NaN,NaN
8,Czechia,"198,775.27","180,859.67","175,034.85","168,323.67","159,410.64","158,371.95","161,709.65","157,200.37","151,034.59",...,"129,251.71","127,080.04","128,506.45","130,106.97","130,951.94","129,180.06","123,451.26","113,072.05","118,381.69",NaN
9,Denmark,"71,820.94","82,433.00","76,517.58","78,670.86","82,598.77","79,748.73","92,928.69","83,411.41","79,455.94",...,"57,779.39","53,608.90","50,834.62","52,906.15","50,773.91","50,726.02","46,941.67","44,483.80","45,515.58",NaN


In [18]:

ghg.drop(0, inplace=True)


In [5]:
import pandas as pd


# Load the data and skip header rows
ghg = pd.read_csv("/Users/user/Desktop/Individual_DE/Website_data/greenhousegas.csv", skiprows=4)

# Drop the unnecessary second column
ghg.drop(columns=[ghg.columns[1]], inplace=True)

# Drop the first row if it's leftover from formatting
ghg.drop(index=0, inplace=True)

# Rename first column
ghg.rename(columns={ghg.columns[0]: "Country"}, inplace=True)

# Strip any whitespace from column names
ghg.columns = ghg.columns.str.strip()

# ONLY apply float conversion to columns that are actual years
for col in ghg.columns[1:]:
    ghg[col] = ghg[col].astype(str).str.replace(",", "").str.strip()
    ghg[col] = pd.to_numeric(ghg[col], errors="coerce")  # Coerce non-numeric to NaN

# Drop any rows with missing countries (optional)
ghg = ghg[ghg["Country"].notna()]

# Melt to long format
ghg_long = pd.melt(ghg, id_vars="Country", var_name="Year", value_name="Emissions")

# Drop rows with missing emissions
ghg_long = ghg_long.dropna(subset=["Emissions"])

# Convert Year to integer
ghg_long["Year"] = ghg_long["Year"].astype(int)



In [28]:
ghg_long.head(10)

,Country,Year,Emissions
0,Australia,1990,438056.76
1,Austria,1990,79047.23
2,Belgium,1990,145844.47
3,Canada,1990,588602.82
4,Chile,1990,48818.66
5,Colombia,1990,101878.75
6,Costa Rica,1990,8196.56
7,Czechia,1990,198775.27
8,Denmark,1990,71820.94
9,Estonia,1990,40276.36


In [6]:
import sqlite3

conn = sqlite3.connect("website.db")
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS greenhouse_emissions (
    country TEXT,
    year INTEGER,
    emissions REAL,
    FOREIGN KEY (country) REFERENCES Country(country)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS deforestation (
    country TEXT,
    year INTEGER,
    deforestation REAL,
    FOREIGN KEY (country) REFERENCES Country(country)
)
''')

conn.commit()



In [7]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)


[('GreenhouseGasEmissions',), ('Deforestation',), ('greenhouse_emissions',), ('Country',)]


In [10]:

unique_countries = pd.concat([ghg_long['Country'], cdf_long['Country']]).dropna().unique()
countries = pd.DataFrame(unique_countries, columns=["country_name"])
countries.to_sql("Country", conn, if_exists="replace", index=False)

238

In [11]:
cursor.execute("DROP TABLE IF EXISTS greenhouse_emissions")
conn.commit()

In [12]:
ghg_long.to_sql("GreenhouseGasEmissions", conn, if_exists="replace", index=False)
cdf_long.to_sql("Deforestation", conn, if_exists="replace", index=False)



6467

In [13]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)

[('Country',), ('GreenhouseGasEmissions',), ('Deforestation',)]


In [14]:


conn = sqlite3.connect("website.db")
cursor = conn.cursor()
cursor.execute("SELECT DISTINCT country FROM GreenhouseGasEmissions")
countries = [row[0] for row in cursor.fetchall()]
print(countries)
conn.close()


['Australia', 'Austria', 'Belgium', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Japan', 'Korea', 'Latvia', 'Lithuania', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland', 'Portugal', 'Slovak Republic', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'United Kingdom', 'United States', 'European Union (28 countries)', '·\u2007\u2007Argentina', '·\u2007\u2007Bulgaria', '·\u2007\u2007Croatia', '·\u2007\u2007Cyprus', '·\u2007\u2007Kazakhstan', '·\u2007\u2007Malta', '·\u2007\u2007Romania', '·\u2007\u2007Russia', '·\u2007\u2007Ukraine', 'Israel', '·\u2007\u2007South Africa']


In [15]:
import sqlite3

# Connect to your SQLite database file
conn = sqlite3.connect('website.db')
cursor = conn.cursor()

# Country to test
country = "Russian Federation"

# SQL query
query = "SELECT * FROM Deforestation WHERE country = ?"
cursor.execute(query, (country,))

# Fetch all results
rows = cursor.fetchall()

# Check if any data was returned
if rows:
    print(f"Data found for {country}:")
    for row in rows:
        print(row)
else:
    print(f"No data found for {country}")

# Close the connection
conn.close()


Data found for Russian Federation:
('Russian Federation', '1990', 511425.0553)
('Russian Federation', '1991', 492538.3914)
('Russian Federation', '1992', 530146.934)
('Russian Federation', '1993', 494550.9447)
('Russian Federation', '1994', 463817.7666)
('Russian Federation', '1995', 445610.9937)
('Russian Federation', '1996', 418414.7397)
('Russian Federation', '1997', 398894.6581)
('Russian Federation', '1998', 381260.3302)
('Russian Federation', '1999', 380713.1631)
('Russian Federation', '2000', 388011.408)
('Russian Federation', '2001', 353293.4973)
('Russian Federation', '2002', 349358.5815)
('Russian Federation', '2003', 353680.5437)
('Russian Federation', '2004', 352121.3146)
('Russian Federation', '2005', 350643.8906)
('Russian Federation', '2006', 353798.4294)
('Russian Federation', '2007', 360783.0196)
('Russian Federation', '2008', 369634.5239)
('Russian Federation', '2009', 362664.4384)
('Russian Federation', '2010', 369082.4464)
('Russian Federation', '2011', 400979.6005)

In [16]:

# Connect to your SQLite database file
conn = sqlite3.connect('website.db')
cursor = conn.cursor()

# Country to test
country = "Russia"

# SQL query
query = "SELECT * FROM GreenhouseGasEmissions WHERE country = ?"
cursor.execute(query, (country,))

# Fetch all results
rows = cursor.fetchall()

# Check if any data was returned
if rows:
    print(f"Data found for {country}:")
    for row in rows:
        print(row)
else:
    print(f"No data found for {country}")

# Close the connection
conn.close()

No data found for Russia


In [23]:
import sqlite3

# Connect to your database
conn = sqlite3.connect('website.db')
cursor = conn.cursor()

# Run the query
cursor.execute("SELECT DISTINCT Country FROM GreenhouseGasEmissions")
results = cursor.fetchall()

# Flatten and print the list
countries = [row[0] for row in results]
print(countries)




['Australia', 'Austria', 'Belgium', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Japan', 'Korea', 'Latvia', 'Lithuania', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland', 'Portugal', 'Slovak Republic', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'United Kingdom', 'United States', 'European Union (28 countries)', '·\u2007\u2007Argentina', '·\u2007\u2007Bulgaria', '·\u2007\u2007Croatia', '·\u2007\u2007Cyprus', '·\u2007\u2007Kazakhstan', '·\u2007\u2007Malta', '·\u2007\u2007Romania', '·\u2007\u2007Russia', '·\u2007\u2007Ukraine', 'Israel', '·\u2007\u2007South Africa']


In [17]:
conn = sqlite3.connect('website.db')

cursor = conn.cursor()

cursor.execute("DELETE FROM Country WHERE country_name = ?", ("European Union (28 countries)",))


In [1]:
import sqlite3

# Mapping: old names → new names
country_corrections = {
    '·\u2007\u2007Russia': 'Russian Federation',
    '·\u2007\u2007Argentina': 'Argentina',
    '·\u2007\u2007Bulgaria': 'Bulgaria',
    '·\u2007\u2007Croatia': 'Croatia',
    '·\u2007\u2007Cyprus': 'Cyprus',
    '·\u2007\u2007Kazakhstan': 'Kazakhstan',
    '·\u2007\u2007Malta': 'Malta',
    '·\u2007\u2007Romania': 'Romania',
    '·\u2007\u2007Ukraine': 'Ukraine',
    '·\u2007\u2007South Africa': 'South Africa',
    'Türkiye': 'Turkey',
    'Slovak Republic': 'Slovakia'
}

conn = sqlite3.connect('website.db')
cursor = conn.cursor()

# Perform updates
for old_name, new_name in country_corrections.items():
    cursor.execute("""
        UPDATE GreenhouseGasEmissions
        SET Country = ?
        WHERE Country = ?
    """, (new_name, old_name))
    print(f"Updated: '{old_name}' → '{new_name}'")

conn.commit()
conn.close()


Updated: '·  Russia' → 'Russian Federation'
Updated: '·  Argentina' → 'Argentina'
Updated: '·  Bulgaria' → 'Bulgaria'
Updated: '·  Croatia' → 'Croatia'
Updated: '·  Cyprus' → 'Cyprus'
Updated: '·  Kazakhstan' → 'Kazakhstan'
Updated: '·  Malta' → 'Malta'
Updated: '·  Romania' → 'Romania'
Updated: '·  Ukraine' → 'Ukraine'
Updated: '·  South Africa' → 'South Africa'
Updated: 'Türkiye' → 'Turkey'
Updated: 'Slovak Republic' → 'Slovakia'


In [3]:
import sqlite3

conn = sqlite3.connect('website.db', timeout=10)  # Add timeout to reduce locking issues
cursor = conn.cursor()

try:
    # Begin a transaction
    cursor.execute("BEGIN")

    # Get all rows
    cursor.execute("SELECT rowid, country_name FROM Country")
    rows = cursor.fetchall()

    for rowid, name in rows:
        # Clean the name
        clean_name = name.replace('·', '').replace('\u2007', '').strip()

        if clean_name != name:
            cursor.execute("UPDATE Country SET country_name = ? WHERE rowid = ?", (clean_name, rowid))
            print(f"Updated: '{name}' → '{clean_name}'")

    # Commit the whole transaction
    conn.commit()

except sqlite3.OperationalError as e:
    print(f"SQLite error: {e}")
    conn.rollback()

finally:
    conn.close()



In [4]:
import sqlite3

# Mapping: old names → new names
country_corrections = {
    '·\u2007\u2007Russia': 'Russian Federation',
    '·\u2007\u2007Argentina': 'Argentina',
    '·\u2007\u2007Bulgaria': 'Bulgaria',
    '·\u2007\u2007Croatia': 'Croatia',
    '·\u2007\u2007Cyprus': 'Cyprus',
    '·\u2007\u2007Kazakhstan': 'Kazakhstan',
    '·\u2007\u2007Malta': 'Malta',
    '·\u2007\u2007Romania': 'Romania',
    '·\u2007\u2007Ukraine': 'Ukraine',
    '·\u2007\u2007South Africa': 'South Africa',
    'Türkiye': 'Turkey',
    'Slovak Republic': 'Slovakia'
}

conn = sqlite3.connect('website.db')
cursor = conn.cursor()

# Perform updates
for old_name, new_name in country_corrections.items():
    cursor.execute("""
        UPDATE Country
        SET country_name = ?
        WHERE country_name = ?
    """, (new_name, old_name))
    print(f"Updated: '{old_name}' → '{new_name}'")

conn.commit()
conn.close()


Updated: '·  Russia' → 'Russian Federation'
Updated: '·  Argentina' → 'Argentina'
Updated: '·  Bulgaria' → 'Bulgaria'
Updated: '·  Croatia' → 'Croatia'
Updated: '·  Cyprus' → 'Cyprus'
Updated: '·  Kazakhstan' → 'Kazakhstan'
Updated: '·  Malta' → 'Malta'
Updated: '·  Romania' → 'Romania'
Updated: '·  Ukraine' → 'Ukraine'
Updated: '·  South Africa' → 'South Africa'
Updated: 'Türkiye' → 'Turkey'
Updated: 'Slovak Republic' → 'Slovakia'


In [ ]:
def generate_plot(country):
    conn = get_connection()

    ghg = pd.read_sql("SELECT Year, Emissions FROM GreenhouseGasEmissions WHERE Country = ?", conn, params=(country,))
    defo = pd.read_sql("SELECT Year, Deforestation FROM Deforestation WHERE Country = ?", conn, params=(country,))
    conn.close()
    ghg["Year"] = ghg["Year"].astype(int)
    defo["Year"] = defo["Year"].astype(int)

    if ghg.empty or defo.empty:
     return None  # or return a placeholder message/image

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))  # two plots side by side

    # Greenhouse Gas Emissions plot
    ax1.plot(ghg["Year"], ghg["Emissions"], color="green")
    ax1.set_title("Greenhouse Gas Emissions")
    ax1.set_xlabel("Year")
    ax1.set_ylabel("Emissions")
    ax1.set_xticks(np.arange(min(ghg["Year"]), max(ghg["Year"])+1, 3))
    ax1.tick_params(axis='x', rotation=45)

    # Deforestation plot
    ax2.plot(defo["Year"], defo["Deforestation"], color="brown")
    ax2.set_title("Deforestation")
    ax2.set_xlabel("Year")
    ax2.set_ylabel("Hectares Lost")
    ax2.set_xticks(np.arange(min(defo["Year"]), max(defo["Year"])+1, 3))
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()

    img = io.BytesIO()
    plt.savefig(img, format='png')
    img.seek(0)
    plot_url = base64.b64encode(img.getvalue()).decode()
    plt.close()

    return f"data:image/png;base64,{plot_url}"